# v3 GSM8K-Hindi full eval (n=100)

Same protocol as `scripts/bench_gsm_hi.py`: 4-shot CoT, greedy, exact match.
Runs on Kaggle T4 (MPS kept timing out locally). Model: `eulogik/Bharat-Tiny-LLM-v3` fp16.
Saves `bench_gsm_hi_v3_full.json` to /kaggle/working for `kernels output` download.

In [ ]:
#@title 0 - setup
import subprocess, sys
pkgs = ["transformers>=4.53,<5", "accelerate>=1.0", "datasets>=3.0",
        "safetensors", "huggingface_hub", "sentencepiece"]
subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", *pkgs])
subprocess.call([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"],
                stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
import transformers, torch
print("transformers", transformers.__version__)
print(f"torch {torch.__version__} cuda={torch.cuda.is_available()} gpus={torch.cuda.device_count()}")

# Tokenizer workaround: v3 tokenizer_config has extra_special_tokens as a list;
# transformers Qwen2-fast init expects a dict (same fix as v34 kernel)
from transformers.tokenization_utils_base import PreTrainedTokenizerBase
if not getattr(PreTrainedTokenizerBase, "_v3eval_init_wrapped", False):
    _orig_init = PreTrainedTokenizerBase.__init__
    def _v3eval_init(self, *args, **kwargs):
        est = kwargs.get("extra_special_tokens")
        if isinstance(est, (list, tuple, set)):
            kwargs["extra_special_tokens"] = {}
        return _orig_init(self, *args, **kwargs)
    PreTrainedTokenizerBase.__init__ = _v3eval_init
    PreTrainedTokenizerBase._v3eval_init_wrapped = True
    print("tokenizer extra_special_tokens workaround installed")

In [ ]:
#@title 1 - config + load model
import os, json, time
from pathlib import Path
WORK = Path("/kaggle/working/v3eval"); WORK.mkdir(parents=True, exist_ok=True)
MODEL = "eulogik/Bharat-Tiny-LLM-v3"
N = 100
MAX_TOKENS = 320
NUM_SHOTS = 4
OUT = WORK / "bench_gsm_hi_v3_full.json"
print("model:", MODEL, "n:", N)

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
t0 = time.time()
tok = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL, dtype=torch.float16, device_map={"": 0}, trust_remote_code=True)
model.eval()
print(f"loaded in {time.time()-t0:.1f}s on {next(model.parameters()).device}")

In [ ]:
#@title 2 - eval (4-shot CoT greedy exact-match, heartbeat + incremental save)
import re
from datasets import load_dataset
INSTR = ("\u0928\u093f\u092e\u094d\u0928\u0932\u093f\u0916\u093f\u0924 \u0917\u0923\u093f\u0924 \u0915\u0940 \u0938\u092e\u0938\u094d\u092f\u093e\u0913\u0902 \u0915\u094b \u0939\u0932 \u0915\u0930\u0947\u0902\u0964 \u091a\u0930\u0923-\u0926\u0930-\u091a\u0930\u0923 \u0938\u094b\u091a\u0947\u0902 \u0914\u0930 \u0939\u0930 \u0909\u0924\u094d\u0924\u0930 \u0915\u0947 \u0905\u0902\u0924 \u092e\u0947\u0902 '\u0909\u0924\u094d\u0924\u0930: <\u0938\u0902\u0916\u094d\u092f\u093e>' \u0932\u093f\u0916\u0947\u0902\u0964\n\n")

def compact_answer(ans):
    m = re.search(r"#{3,4}\s*(-?[\d,]+(?:\.\d+)?)", ans)
    final = m.group(1).replace(",", "") if m else ""
    body = re.sub(r"<<[^>]*>>", "", ans.split("####")[0]).strip()
    lines = [l.strip() for l in body.splitlines() if l.strip()]
    keep = lines[-NUM_SHOTS:] if len(lines) > NUM_SHOTS else lines
    return "\n".join(keep) + f"\n\u0909\u0924\u094d\u0924\u0930: {final}" if final else body

def build_prompt(exemplars, question):
    parts = [INSTR]
    for ex in exemplars:
        parts.append(f"\u092a\u094d\u0930\u0936\u094d\u0928: {ex['question']}\n{compact_answer(ex['answer'])}\n")
    parts.append(f"\u092a\u094d\u0930\u0936\u094d\u0928: {question}")
    return "\n".join(parts)

def extract_answer(text):
    ms = re.findall(r"\u0909\u0924\u094d\u0924\u0930[:\uff1a=\s]*(-?[\d,]+(?:\.\d+)?)", text)
    if ms: return ms[-1].replace(",", "").rstrip(".")
    m = re.search(r"####\s*(-?[\d,]+(?:\.\d+)?)", text)
    if m: return m.group(1).replace(",", "")
    nums = re.findall(r"-?\d[\d,]*(?:\.\d+)?", text)
    return nums[-1].replace(",", "").rstrip(".") if nums else None

ds = load_dataset("bingbangboom/gsm8k-hindi")
train, test = ds["train_main"], ds["test_main"]
exemplars = [train[i] for i in range(NUM_SHOTS)]
print(f"test rows: {len(test)}", flush=True)

correct, t0, results = 0, time.time(), []
for i, row in enumerate(test.select(range(N))):
    prompt = build_prompt(exemplars, row["question"])
    chat = [{"role": "user", "content": prompt}]
    try:
        rendered = tok.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
        inputs = tok(rendered, return_tensors="pt", truncation=True, max_length=1536).to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=MAX_TOKENS, do_sample=False,
                                 pad_token_id=tok.eos_token_id)
        text = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    except Exception as e:
        text = ""
        print(f"  [{i}] gen error: {e}", flush=True)
    pred = extract_answer(text)
    gm = re.search(r"#{3,4}\s*(-?[\d,]+(?:\.\d+)?)", row["answer"])
    if gm: gold = gm.group(1).replace(",", "")
    else:
        tail = re.findall(r"-?[\d][\d,]*(?:\.\d+)?", row["answer"])
        gold = tail[-1].replace(",", "") if tail else None
    try: ok = gold is not None and pred is not None and abs(float(pred)-float(gold)) < 1e-6
    except ValueError: ok = False
    correct += int(ok)
    results.append({"i": i, "pred": pred, "gold": gold, "ok": ok,
                              "text": text[:300]})
    if (i+1) % 10 == 0:
        print(f"  {i+1}/{N}: acc={correct/(i+1)*100:.1f}%", flush=True)
        (WORK / "bench_partial.json").write_text(json.dumps(
            {"done": i+1, "correct": correct, "results": results}, ensure_ascii=False))

elapsed = time.time() - t0
summary = {"model": "eulogik/Bharat-Tiny-LLM-v3", "accuracy_pct": round(correct/N*100, 2),
           "correct": correct, "total": N, "time_sec": round(elapsed, 1),
           "protocol": "4-shot CoT, greedy, exact match (same as scripts/bench_gsm_hi.py)"}
print(json.dumps(summary, indent=2), flush=True)
OUT.write_text(json.dumps({"benchmark": "GSM8K-Hindi", "shots": NUM_SHOTS,
                           "decoding": "greedy", "max_tokens": MAX_TOKENS,
                           "summary": summary, "results": results},
                          indent=2, ensure_ascii=False))
print("saved", OUT, flush=True)

In [ ]:
#@title 3 - status
import os
print("STATE files:")
for root, dirs, files in os.walk("/kaggle/working/v3eval"):
    for f in files:
        p = os.path.join(root, f)
        print(f"  {p} ({os.path.getsize(p)} bytes)")